# 01 - QA Dataset Creation

The assignment provides **no QA dataset**, so we build one from the radiology
reports. For each report, Google Gemini generates question-answer pairs that
are **grounded strictly in the report text** (three styles: factual yes/no,
location/severity, impression).

The result (`data/qa/qa_dataset.json`) serves as the evaluation set for the
RAG QA mode. This notebook is the documented, reproducible build process.

**Prerequisites:** `pip install -e .` + `pip install -r requirements.txt`,
a `.env` with `GEMINI_API_KEY`, and a downloaded dataset (`scripts/download_data.py`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from cxr.data.loader import dataset_summary, load_records
from cxr.data.qa_builder import generate_qa_for_report, build_qa_dataset, load_qa_dataset

dataset_summary()

## Step 1 - Sanity check: generate QA pairs for a single report

In [ ]:
sample = load_records(limit=1)[0]
print('REPORT:\n', sample.report_text, '\n')

for pair in generate_qa_for_report(sample.report_text):
    print(f"[{pair['qtype']}]")
    print('  Q:', pair['question'])
    print('  A:', pair['answer'])

## Step 2 - Build the full QA dataset

Capped by `qa_generation.max_reports` in `config.yaml`. `sleep` paces requests
to respect the Gemini free-tier rate limit.

In [ ]:
out_path = build_qa_dataset()
out_path

## Step 3 - Inspect the generated dataset

In [ ]:
import pandas as pd

qa = load_qa_dataset()
df = pd.DataFrame(qa)
print(f'Total QA pairs: {len(df)}')
print('\nPer question type:')
print(df['qtype'].value_counts())
df[['qtype', 'question', 'answer']].head(10)